# C16_E04 — Edge AI: inferencia local con modelo ligero

**Capítulo:** 16 — Machine Learning, Edge AI y supervisión inteligente de lazos  
**Identificador:** `C16_E04`  
**Objetivo pedagógico:** Mostrar el patrón Edge AI: modelo entrenado offline, inferencia local con latencia controlada.  
**Librerías:** scikit-learn, numpy

## Ejemplo industrial

Pasarela industrial con detector de anomalías ejecutando localmente sin dependencia de la nube.

---

> **Aviso de uso responsable.** Este notebook es didáctico. No es código de producción. Cualquier implementación real requiere validación independiente. Ver `docs/politica_uso_responsable.md`.

## 1. Modelo ligero entrenado offline y exportado para Edge

In [1]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import time
np.random.seed(0)

# Dataset entrenamiento (telemetría histórica)
N = 5000
X = np.random.normal(0, 1, size=(N, 4))
y = (X[:,0] + X[:,1] - X[:,2] > 0).astype(int)

pipeline = Pipeline([
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(max_iter=200))
])
pipeline.fit(X, y)
print("Acc en train:", pipeline.score(X, y))

Acc en train: 0.9974


## 2. Inferencia local con presupuesto de latencia

In [2]:
# Simulación de inferencia en pasarela Edge
N_inf = 1000
X_run = np.random.normal(0, 1, size=(N_inf, 4))
t0 = time.perf_counter()
preds = pipeline.predict(X_run)
t1 = time.perf_counter()
lat = (t1 - t0)/N_inf*1e6
print(f"Latencia media: {lat:.2f} µs/inferencia")
print(f"Throughput estimado: {1e6/lat:.0f} inferencias/segundo")

Latencia media: 0.39 µs/inferencia
Throughput estimado: 2588307 inferencias/segundo


## 3. Interpretación

Un pipeline simple (escalado + regresión logística) ejecuta inferencias en pocos microsegundos en CPU estándar. Es adecuado para Edge AI en pasarelas industriales con presupuesto de latencia milisegundo. **Buenas prácticas obligatorias** para Edge: monitorización de drift en cada dispositivo, procedimiento de actualización OTA con verificación, capacidad de rollback, gestión centralizada de versiones. Modelos más complejos (redes neuronales) pueden requerir cuantización o aceleración (TensorRT, ONNX, OpenVINO).